# LaunchPad — Google Colab deployment

Deploy the existing LaunchPad website using your own AWS account. This notebook installs the tools, deploys the CDK stack, uploads the frontend to private S3, and publishes it through CloudFront HTTPS.

**Before running:** use a trusted private Colab runtime. Never paste AWS access keys into chat or commit them to GitHub. AWS resources may incur charges.

## 1. Configure AWS credentials securely

Create an IAM principal with only the permissions needed for this deployment (or use your organization's approved deployment role). Prefer short-lived credentials. The cells below prompt privately and do not print the secret values. Do not share this notebook with credentials embedded.

In [ ]:
import os, getpass
os.environ['AWS_ACCESS_KEY_ID'] = getpass.getpass('AWS Access Key ID: ')
os.environ['AWS_SECRET_ACCESS_KEY'] = getpass.getpass('AWS Secret Access Key: ')
token = getpass.getpass('AWS Session Token (press Enter if not required): ')
if token:
    os.environ['AWS_SESSION_TOKEN'] = token
else:
    os.environ.pop('AWS_SESSION_TOKEN', None)
region = input('AWS region [ap-south-1]: ').strip() or 'ap-south-1'
os.environ['AWS_DEFAULT_REGION'] = region
print('Credentials loaded for this Colab runtime. Secret values were not displayed.')

## 2. Install deployment tools and download your repository

In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y nodejs npm
!python -m pip -q install --upgrade awscli
!node --version
!npm --version
!aws --version
!git clone https://github.com/leelasreeamrutha99-glitch/LaunchPad-Student-Opportunity-Platform.git /content/LaunchPad-Student-Opportunity-Platform || (cd /content/LaunchPad-Student-Opportunity-Platform && git pull --ff-only)
%cd /content/LaunchPad-Student-Opportunity-Platform

## 3. Verify AWS identity and install project dependencies

The account identifier is shown only in your private notebook output. Confirm it is the AWS account where you intend to create resources.

In [ ]:
import json, subprocess
identity = json.loads(subprocess.check_output(['aws','sts','get-caller-identity','--output','json'], text=True))
print('Authenticated AWS account:', identity['Account'])
print('Authenticated principal:', identity['Arn'])
%cd /content/LaunchPad-Student-Opportunity-Platform/infra
!npm install
%cd /content/LaunchPad-Student-Opportunity-Platform/backend
!npm install --omit=dev
%cd /content/LaunchPad-Student-Opportunity-Platform/infra
!npm run build

## 4. Bootstrap and deploy the AWS infrastructure

This creates the S3 website bucket, CloudFront distribution, API Gateway, Lambda function and DynamoDB table. Review the resources and charges in AWS before proceeding.

In [ ]:
!npx cdk bootstrap
!npx cdk deploy --require-approval never --outputs-file cdk-outputs.json

## 5. Upload the website and configure its API health check

In [ ]:
import json, pathlib, subprocess
outputs = json.loads(pathlib.Path('cdk-outputs.json').read_text())
stack = outputs['LaunchpadWellArchitectedStack']
bucket = stack['FrontendBucketName']
site_url = stack['CloudFrontUrl']
api_url = stack['ApiUrl']
distribution_id = stack['CloudFrontDistributionId']
print('Website:', site_url)
print('API health endpoint:', api_url.rstrip('/') + '/health')
config = 'window.LAUNCHPAD_CONFIG = {\\n  API_BASE_URL: ' + json.dumps(api_url) + '\\n};\\n'
pathlib.Path('../web/config.js').write_text(config)
subprocess.run(['aws','s3','sync','../web',f's3://{bucket}','--delete'],check=True)
subprocess.run(['aws','cloudfront','create-invalidation','--distribution-id',distribution_id,'--paths','/*'],check=True)
print('Frontend uploaded and CloudFront invalidation requested.')

## 6. Verify the public website and API

CloudFront may take several minutes to finish deploying. If the first check fails, wait a few minutes and rerun this cell.

In [ ]:
import time, urllib.request
for url in [site_url, api_url.rstrip('/') + '/health']:
    try:
        with urllib.request.urlopen(url, timeout=30) as response:
            print(response.status, url)
            print(response.read(500).decode('utf-8','replace'))
    except Exception as exc:
        print('Not ready yet:', url, repr(exc))
print('Public website URL:', site_url)

## Important project limitations and cleanup

- The current UI runs in Demo Mode and stores saved items/application status in each browser's local storage. The API health endpoint confirms connectivity, but the UI does **not** yet synchronize those records to DynamoDB.
- The backend currently has no user authentication or per-user data isolation. Do not store real personal or sensitive data in it; do not advertise it as production-ready.
- AWS resources can incur ongoing charges. CloudFront, Lambda/API requests, DynamoDB, S3 storage and data transfer are billed according to AWS pricing.
- The S3 bucket and DynamoDB table use retention protections. Deleting the CDK stack will not necessarily delete retained data resources. Review the CloudFormation stack and resources before cleanup.
- Never publish your AWS credentials, session token, account ID, or private deployment outputs in a public repository or screenshot.